# Day 15 - MERGE and UPSERT Pattern

**Topic:** MERGE and UPSERT Pattern  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกใช้ `MERGE INTO` เพื่อทำ update/insert records เข้า Lakehouse table ด้วย business key และ incremental update batch

วันนี้เราจะเรียน pattern ที่สำคัญมากของ Lakehouse pipeline คือ **MERGE / UPSERT** โดยจะเริ่มจาก target table ที่มีข้อมูลเดิม แล้วรับ source update batch เข้ามา จากนั้นใช้ `MERGE INTO` เพื่อ update records ที่ match และ insert records ที่ยังไม่มีใน target

## Goal of the day

1. อธิบายได้ว่า **MERGE** และ **UPSERT** ต่างจาก `append` อย่างไร
2. เข้าใจ role ของ **business key** ในการจับคู่ record ระหว่าง source และ target
3. ใช้ Spark SQL `MERGE INTO` เพื่อ update/insert ข้อมูลใน Lakehouse table ได้
4. ตรวจ source data รอบใหม่ก่อน merge เช่น duplicate keys และแยกได้ว่า record ไหนจะถูก update หรือ insert
5. เริ่มระวัง idempotency และ rerun safety เบื้องต้นของ incremental pipeline

## Why it matters in production

ใน production pipeline เราไม่ได้รับข้อมูลแบบ append-only เสมอไป หลายครั้ง source จะส่งข้อมูลที่มีทั้ง record ใหม่และ record เดิมที่ถูกแก้ไข เช่น customer profile, product master, policy status หรือ transaction correction

ถ้าใช้ `append` อย่างเดียว:

- record เดิมจะไม่ถูกแก้ไข แต่จะมี row ใหม่ถูกเพิ่มเข้ามาแทน
- business key เดียวกัน เช่น `customer_id` อาจมีหลาย rows ใน target table
- downstream report อาจนับซ้ำ หรือหยิบค่าที่ไม่ใช่ค่าล่าสุดไปใช้
- ถ้า rerun pipeline ด้วย batch เดิม อาจ append ข้อมูลชุดเดิมซ้ำเข้าไปอีก

`MERGE` ช่วยให้เราจัดการ update/insert ใน operation เดียว โดยใช้ business key เป็นตัวตัดสินว่า record นั้นควรถูก update หรือ insert

Production takeaway:

> MERGE ไม่ใช่แค่ syntax สำหรับ update ข้อมูล แต่เป็น pattern สำคัญของ incremental load, CDC, dimension update และ rerun-safe Lakehouse pipeline

## Key concepts

| Concept | Meaning |
|---|---|
| MERGE | คำสั่งที่รวม logic ของ update, insert และ delete ตามเงื่อนไขที่กำหนดไว้ใน operation เดียว |
| UPSERT | แนวคิด update ถ้ามี record อยู่แล้ว และ insert ถ้ายังไม่มี record |
| Target table | table ปลายทางที่เก็บข้อมูลปัจจุบัน เช่น customer profile table |
| Source updates | batch ข้อมูลใหม่/เปลี่ยนแปลงที่เข้ามาจาก source system |
| Business key | key ที่ใช้ identify record ในเชิง business เช่น `customer_id`, `product_id`, `policy_id` |
| Matched record | record ใน source ที่หา key เจอใน target จึงมักถูก update |
| Not matched record | record ใน source ที่ยังไม่มี key ใน target จึงมักถูก insert |
| Idempotency | rerun ด้วย input เดิมแล้ว target table ยังได้ผลลัพธ์เหมือนเดิม ไม่เพิ่มข้อมูลซ้ำหรือทำให้ข้อมูลเสีย |
| Duplicate source key | source batch มี key ซ้ำมากกว่า 1 row ซึ่งอาจทำให้ merge ผิด logic หรือ fail ได้ |

## Concept explanation

### MERGE / UPSERT คืออะไร?

ใน Day 14 เราเรียนเรื่อง `append` และ `overwrite` แล้ว วันนี้เราจะเพิ่ม pattern ที่ใช้จริงใน incremental pipeline คือ `MERGE`

แนวคิดหลักคือ:

```sql
MERGE INTO target
USING source
ON target.business_key = source.business_key
WHEN MATCHED THEN UPDATE
WHEN NOT MATCHED THEN INSERT
```

แปลเป็นภาษาง่าย ๆ:

> ถ้า source key เจอใน target → update row เดิม  
> ถ้า source key ไม่เจอใน target → insert row ใหม่

### ทำไม append อย่างเดียวไม่พอ?

สมมติ target มี customer `C002` อยู่แล้ว แต่ source ส่ง email ใหม่ของ `C002` เข้ามา

ถ้าใช้ append จะได้ `C002` สอง rows ใน target table ทำให้ downstream ต้องมี logic เพิ่มเพื่อเลือก record ล่าสุด เช่น sort ด้วย `updated_at` แล้วเลือก row ล่าสุดต่อ `customer_id`

แต่ถ้า target table นี้ตั้งใจให้เป็น current customer profile การใช้ merge จะเหมาะกว่า เพราะสามารถ update `C002` row เดิมให้เป็น email ล่าสุด และ insert customer ใหม่ที่ยังไม่มีใน target ได้ใน operation เดียว

### Business key สำคัญยังไง?

`MERGE` ต้องมีเงื่อนไข `ON` เพื่อบอกว่า source row และ target row เป็น record เดียวกันหรือไม่

ตัวอย่าง:

```sql
ON target.customer_id = source.customer_id
```

ใน production ต้องเลือก key ให้ถูก เพราะถ้า key กว้างเกินไปหรือแคบเกินไป อาจ update ผิด row หรือ insert record ซ้ำ

### Source update batch ควร unique ต่อ business key

ก่อน merge ควรตรวจว่า source batch มี duplicate key หรือไม่ เช่น `customer_id` เดียวกันมา 2 rows ใน batch เดียว

ถ้ามี duplicate key ต้องตัดสินใจก่อนว่า:

- จะ keep latest record ด้วย `updated_at`
- จะ reject/quarantine duplicate source keys
- หรือจะ aggregate/resolve conflict ด้วย business rule อื่น

ใน notebook นี้เราจะใช้ pattern ง่าย ๆ คือ keep latest record ต่อ `customer_id` ก่อน merge


## Mermaid diagram: MERGE / UPSERT Flow

```mermaid
flowchart LR
    A[Source update batch] --> B[Check duplicate business keys]
    B --> C[Keep latest record per key]
    C --> D{customer_id exists in target?}
    T[(Target Delta table)] --> D
    D -- Yes / matched --> E[UPDATE existing row]
    D -- No / not matched --> F[INSERT new row]
    E --> G[(Upserted target table)]
    F --> G
```

Key idea:

> MERGE ควรเริ่มจาก source ที่ clean พอสมควร และมี business key ชัดเจน ไม่ใช่โยน batch ดิบที่ key ซ้ำเข้า target ทันที

## Setup / imports

In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 3, Finished, Available, Finished, False)

## Create mock data

เราจะสร้างข้อมูล 2 ชุด:

1. `day15_customer_profile` = target Lakehouse table ที่มี customer profile เดิม
2. `df_customer_updates_raw` = source update batch ที่มีทั้ง update, insert และ duplicate source key เพื่อให้ฝึกตรวจข้อมูลก่อน merge

In [3]:
target_table_name = "day15_customer_profile"
source_view_name = "day15_customer_updates"

customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

initial_customer_data = [
    ("C001", "Anan", "anan.old@example.com", "active", "2025-04-01 09:00:00", "crm"),
    ("C002", "Ben", "ben.old@example.com", "active", "2025-04-01 10:00:00", "crm"),
    ("C003", "Chaba", "chaba@example.com", "active", "2025-04-02 11:00:00", "crm"),
]

df_initial_customers = (
    spark.createDataFrame(initial_customer_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
    .withColumn("last_merge_action", F.lit("INITIAL_LOAD"))
    .withColumn("merge_processed_at", F.current_timestamp())
)

# Create a repeatable lab target table.
(
    df_initial_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table_name)
)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 5, Finished, Available, Finished, False)

+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+
|customer_id|customer_name|email               |status|updated_at         |source_system|last_merge_action|merge_processed_at        |
+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+
|C001       |Anan         |anan.old@example.com|active|2025-04-01 09:00:00|crm          |INITIAL_LOAD     |2026-06-02 07:09:28.375728|
|C002       |Ben          |ben.old@example.com |active|2025-04-01 10:00:00|crm          |INITIAL_LOAD     |2026-06-02 07:09:28.375728|
|C003       |Chaba        |chaba@example.com   |active|2025-04-02 11:00:00|crm          |INITIAL_LOAD     |2026-06-02 07:09:28.375728|
+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+



Source update batch ด้านล่างตั้งใจใส่ `C002` ซ้ำ 2 rows เพื่อให้เห็นว่าเราควร resolve duplicate source key ก่อน merge

In [4]:
source_update_data = [
    # Older update for C002. This should not be used after latest-record selection.
    ("C002", "Ben", "ben.mid@example.com", "active", "2025-04-15 08:00:00", "crm"),
    # Latest update for C002. This should win.
    ("C002", "Ben", "ben.new@example.com", "active", "2025-04-18 08:00:00", "crm"),
    # Existing customer with status change.
    ("C003", "Chaba", "chaba@example.com", "inactive", "2025-04-19 09:30:00", "crm"),
    # New customers.
    ("C004", "Dao", "dao@example.com", "active", "2025-04-20 13:00:00", "mobile_app"),
    ("C005", "Ek", "ek@example.com", "active", "2025-04-20 14:00:00", "mobile_app"),
]

df_customer_updates_raw = (
    spark.createDataFrame(source_update_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_customer_updates_raw.orderBy("customer_id", "updated_at").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 6, Finished, Available, Finished, False)

+-----------+-------------+-------------------+--------+-------------------+-------------+
|customer_id|customer_name|email              |status  |updated_at         |source_system|
+-----------+-------------+-------------------+--------+-------------------+-------------+
|C002       |Ben          |ben.mid@example.com|active  |2025-04-15 08:00:00|crm          |
|C002       |Ben          |ben.new@example.com|active  |2025-04-18 08:00:00|crm          |
|C003       |Chaba        |chaba@example.com  |inactive|2025-04-19 09:30:00|crm          |
|C004       |Dao          |dao@example.com    |active  |2025-04-20 13:00:00|mobile_app   |
|C005       |Ek           |ek@example.com     |active  |2025-04-20 14:00:00|mobile_app   |
+-----------+-------------+-------------------+--------+-------------------+-------------+



## PySpark code examples

ใน section นี้เราจะไล่จากการตรวจ source batch ไปจนถึงการ run `MERGE INTO` จริงกับ Lakehouse table

### Example 1: Check duplicate business keys in source batch

ก่อน merge ต้องรู้ก่อนว่า source batch มี key ซ้ำหรือไม่ เพราะถ้า source มีหลาย rows ที่ match target row เดียวกัน ผลลัพธ์อาจไม่ชัดเจนหรือ merge อาจ fail ได้

In [5]:
df_duplicate_source_keys = (
    df_customer_updates_raw
    .groupBy("customer_id")
    .agg(F.count("*").alias("source_record_count"))
    .filter(F.col("source_record_count") > 1)
)

df_duplicate_source_keys.show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 7, Finished, Available, Finished, False)

+-----------+-------------------+
|customer_id|source_record_count|
+-----------+-------------------+
|C002       |2                  |
+-----------+-------------------+



### Example 2: Keep latest source record per business key

ตัวอย่างนี้ใช้ `row_number()` เพื่อเลือก record ล่าสุดต่อ `customer_id` โดยดูจาก `updated_at` มากสุด

นี่เป็น pattern พื้นฐานที่ใช้บ่อยก่อนทำ merge หรือ latest snapshot update

In [6]:
latest_update_window = Window.partitionBy("customer_id").orderBy(F.col("updated_at").desc())

df_customer_updates = (
    df_customer_updates_raw
    .withColumn("rn", F.row_number().over(latest_update_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_customer_updates.orderBy("customer_id").show(truncate=False)
print("Raw source count:", df_customer_updates_raw.count())
print("Deduplicated source count:", df_customer_updates.count())

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 8, Finished, Available, Finished, False)

+-----------+-------------+-------------------+--------+-------------------+-------------+
|customer_id|customer_name|email              |status  |updated_at         |source_system|
+-----------+-------------+-------------------+--------+-------------------+-------------+
|C002       |Ben          |ben.new@example.com|active  |2025-04-18 08:00:00|crm          |
|C003       |Chaba        |chaba@example.com  |inactive|2025-04-19 09:30:00|crm          |
|C004       |Dao          |dao@example.com    |active  |2025-04-20 13:00:00|mobile_app   |
|C005       |Ek           |ek@example.com     |active  |2025-04-20 14:00:00|mobile_app   |
+-----------+-------------+-------------------+--------+-------------------+-------------+

Raw source count: 5
Deduplicated source count: 4


### Example 3: Preview matched and not matched records before merge

ก่อน run merge จริง เราสามารถใช้ join เพื่อดูได้ว่า source rows ไหนจะ update และ rows ไหนจะ insert

- `left_semi` = source records ที่ match target key → candidate for update
- `left_anti` = source records ที่ไม่ match target key → candidate for insert

In [7]:
df_target_before_merge = spark.read.table(target_table_name)

matched_updates = (
    df_customer_updates.alias("source")
    .join(
        df_target_before_merge.select("customer_id").alias("target"),
        on="customer_id",
        how="left_semi",
    )
)

not_matched_inserts = (
    df_customer_updates.alias("source")
    .join(
        df_target_before_merge.select("customer_id").alias("target"),
        on="customer_id",
        how="left_anti",
    )
)

print("Records expected to update:", matched_updates.count())
matched_updates.orderBy("customer_id").show(truncate=False)

print("Records expected to insert:", not_matched_inserts.count())
not_matched_inserts.orderBy("customer_id").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 9, Finished, Available, Finished, False)

Records expected to update: 2
+-----------+-------------+-------------------+--------+-------------------+-------------+
|customer_id|customer_name|email              |status  |updated_at         |source_system|
+-----------+-------------+-------------------+--------+-------------------+-------------+
|C002       |Ben          |ben.new@example.com|active  |2025-04-18 08:00:00|crm          |
|C003       |Chaba        |chaba@example.com  |inactive|2025-04-19 09:30:00|crm          |
+-----------+-------------+-------------------+--------+-------------------+-------------+

Records expected to insert: 2
+-----------+-------------+---------------+------+-------------------+-------------+
|customer_id|customer_name|email          |status|updated_at         |source_system|
+-----------+-------------+---------------+------+-------------------+-------------+
|C004       |Dao          |dao@example.com|active|2025-04-20 13:00:00|mobile_app   |
|C005       |Ek           |ek@example.com |active|202

### Example 4: Create temp view for source updates

`MERGE INTO` ในตัวอย่างนี้จะใช้ Spark SQL โดยให้ source update batch เป็น temp view

In [8]:
df_customer_updates.createOrReplaceTempView(source_view_name)

spark.sql(f"SELECT * FROM {source_view_name} ORDER BY customer_id").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 10, Finished, Available, Finished, False)

+-----------+-------------+-------------------+--------+-------------------+-------------+
|customer_id|customer_name|email              |status  |updated_at         |source_system|
+-----------+-------------+-------------------+--------+-------------------+-------------+
|C002       |Ben          |ben.new@example.com|active  |2025-04-18 08:00:00|crm          |
|C003       |Chaba        |chaba@example.com  |inactive|2025-04-19 09:30:00|crm          |
|C004       |Dao          |dao@example.com    |active  |2025-04-20 13:00:00|mobile_app   |
|C005       |Ek           |ek@example.com     |active  |2025-04-20 14:00:00|mobile_app   |
+-----------+-------------+-------------------+--------+-------------------+-------------+



### Example 5: Run conditional MERGE / UPSERT

เราจะใช้ conditional update:

```sql
WHEN MATCHED AND source.updated_at > target.updated_at THEN UPDATE
```

เหตุผลคือถ้า source มีข้อมูลเก่ากว่า target ไม่ควร overwrite target ด้วยข้อมูลเก่า และถ้า rerun source batch เดิมซ้ำก็ไม่ควรเปลี่ยนค่าที่ update ไปแล้วโดยไม่จำเป็น

In [9]:
def run_customer_profile_merge(source_view_name: str, target_table_name: str) -> None:
    """Run a basic conditional upsert into the customer profile table."""
    merge_sql = f"""
    MERGE INTO {target_table_name} AS target
    USING {source_view_name} AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED AND source.updated_at > target.updated_at THEN UPDATE SET
        target.customer_name = source.customer_name,
        target.email = source.email,
        target.status = source.status,
        target.updated_at = source.updated_at,
        target.source_system = source.source_system,
        target.last_merge_action = 'UPDATE',
        target.merge_processed_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
        customer_id,
        customer_name,
        email,
        status,
        updated_at,
        source_system,
        last_merge_action,
        merge_processed_at
    ) VALUES (
        source.customer_id,
        source.customer_name,
        source.email,
        source.status,
        source.updated_at,
        source.source_system,
        'INSERT',
        current_timestamp()
    )
    """

    # Execute the MERGE command.
    # This function modifies the target table directly and does not return a DataFrame.
    spark.sql(merge_sql)

run_customer_profile_merge(source_view_name, target_table_name)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 11, Finished, Available, Finished, False)

### Example 6: Validate target table after merge

หลัง merge ควรตรวจอย่างน้อย:

- row count เพิ่มถูกไหม
- records ที่ควร update ถูก update จริงไหม
- records ที่ควร insert ถูก insert จริงไหม
- business key ยัง unique อยู่ไหม

In [10]:
df_target_after_merge = spark.read.table(target_table_name)

df_target_after_merge.orderBy("customer_id").show(truncate=False)

print("Target row count after merge:", df_target_after_merge.count())

print("Check C002 latest email:")
df_target_after_merge.filter(F.col("customer_id") == "C002").show(truncate=False)

print("Check key uniqueness:")
(
    df_target_after_merge
    .groupBy("customer_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
    .show(truncate=False)
)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 12, Finished, Available, Finished, False)

+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|customer_id|customer_name|email               |status  |updated_at         |source_system|last_merge_action|merge_processed_at        |
+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|C001       |Anan         |anan.old@example.com|active  |2025-04-01 09:00:00|crm          |INITIAL_LOAD     |2026-06-02 07:09:28.375728|
|C002       |Ben          |ben.new@example.com |active  |2025-04-18 08:00:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C003       |Chaba        |chaba@example.com   |inactive|2025-04-19 09:30:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C004       |Dao          |dao@example.com     |active  |2025-04-20 13:00:00|mobile_app   |INSERT           |2026-06-02 07:31:27.124448|
|C005       |Ek           |ek@example.com

### Example 7: Rerun the same merge batch and check row count

การ rerun source batch เดิมไม่ควรทำให้ target มี duplicate rows

ในตัวอย่างนี้ `WHEN MATCHED AND source.updated_at > target.updated_at` ช่วยลดการ update ซ้ำเมื่อ source ไม่ใหม่กว่า target

In [11]:
row_count_before_rerun = spark.read.table(target_table_name).count()

run_customer_profile_merge(source_view_name, target_table_name)

row_count_after_rerun = spark.read.table(target_table_name).count()

print("Row count before rerun:", row_count_before_rerun)
print("Row count after rerun:", row_count_after_rerun)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 13, Finished, Available, Finished, False)

Row count before rerun: 5
Row count after rerun: 5
+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|customer_id|customer_name|email               |status  |updated_at         |source_system|last_merge_action|merge_processed_at        |
+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|C001       |Anan         |anan.old@example.com|active  |2025-04-01 09:00:00|crm          |INITIAL_LOAD     |2026-06-02 07:09:28.375728|
|C002       |Ben          |ben.new@example.com |active  |2025-04-18 08:00:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C003       |Chaba        |chaba@example.com   |inactive|2025-04-19 09:30:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C004       |Dao          |dao@example.com     |active  |2025-04-20 13:00:00|mobile_app   |INSERT           |2026-06-02 07:31:2

## Quick recap

| Question | Answer |
|---|---|
| MERGE ใช้ทำอะไร? | ใช้ update/insert target table จาก source update batch |
| UPSERT แปลว่าอะไร? | Update ถ้าเจอ record เดิม และ insert ถ้ายังไม่มี record |
| ใช้ column อะไรเป็นตัว match? | ใช้ business key เช่น `customer_id` |
| ทำไมต้อง deduplicate source ก่อน merge? | เพราะ source key ซ้ำอาจทำให้ update ไม่ชัดเจนหรือ merge fail ได้ |
| `WHEN MATCHED` คืออะไร? | เงื่อนไขสำหรับ source row ที่ match target row |
| `WHEN NOT MATCHED` คืออะไร? | เงื่อนไขสำหรับ source row ที่ยังไม่มีใน target |
| Rerun safety ต้องระวังอะไร? | rerun แล้วต้องไม่ insert ซ้ำ หรือ overwrite target ด้วยข้อมูลเก่า |

## Exercises

### Exercise 1: Merge a second update batch

สร้าง source update batch ชุดที่ 2 แล้ว merge เข้า target table เดิม

Requirements:

- Update `C001` ให้ email ใหม่กว่าเดิม
- Insert customer ใหม่ `C006`
- ใช้ temp view ชื่อ `day15_customer_updates_batch2`
- ใช้ function `run_customer_profile_merge()` ที่สร้างไว้แล้ว

Expected result:

- `C001` ถูก update
- `C006` ถูก insert
- target row count เพิ่มจาก 5 เป็น 6

In [12]:
batch2_view_name = "day15_customer_updates_batch2"

batch2_update_data = [
    ("C001", "Anan", "anan.new@example.com", "active", "2025-04-21 09:00:00", "crm"),
    ("C006", "Fah", "fah@example.com", "active", "2025-04-21 10:00:00", "mobile_app"),
]

df_customer_updates_batch2 = (
    spark.createDataFrame(batch2_update_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_customer_updates_batch2.createOrReplaceTempView(batch2_view_name)

run_customer_profile_merge(batch2_view_name, target_table_name)

spark.read.table(target_table_name).orderBy("customer_id").show(truncate=False)
print("Target row count after batch 2:", spark.read.table(target_table_name).count())

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 14, Finished, Available, Finished, False)

+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|customer_id|customer_name|email               |status  |updated_at         |source_system|last_merge_action|merge_processed_at        |
+-----------+-------------+--------------------+--------+-------------------+-------------+-----------------+--------------------------+
|C001       |Anan         |anan.new@example.com|active  |2025-04-21 09:00:00|crm          |UPDATE           |2026-06-02 07:44:43.903493|
|C002       |Ben          |ben.new@example.com |active  |2025-04-18 08:00:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C003       |Chaba        |chaba@example.com   |inactive|2025-04-19 09:30:00|crm          |UPDATE           |2026-06-02 07:31:27.124448|
|C004       |Dao          |dao@example.com     |active  |2025-04-20 13:00:00|mobile_app   |INSERT           |2026-06-02 07:31:27.124448|
|C005       |Ek           |ek@example.com

### Exercise 2: Try an older update and confirm it does not overwrite newer target data

สร้าง source update ที่มี `updated_at` เก่ากว่า target แล้วลอง merge

Requirements:

- ใช้ `customer_id = C001`
- ตั้ง email เป็นค่าเก่าหรือผิด เช่น `anan.stale@example.com`
- ตั้ง `updated_at` ให้เก่ากว่า target ปัจจุบัน
- merge แล้วตรวจว่า email ของ `C001` ยังเป็นค่าล่าสุดจาก Exercise 1

Expected result:

- `C001` ไม่ถูก overwrite ด้วย stale update

In [13]:
stale_view_name = "day15_customer_updates_stale"

stale_update_data = [
    ("C001", "Anan", "anan.stale@example.com", "active", "2025-04-10 09:00:00", "crm"),
]

df_stale_update = (
    spark.createDataFrame(stale_update_data, customer_schema)
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

df_stale_update.createOrReplaceTempView(stale_view_name)

run_customer_profile_merge(stale_view_name, target_table_name)

spark.read.table(target_table_name).filter(F.col("customer_id") == "C001").show(truncate=False)

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 15, Finished, Available, Finished, False)

+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+
|customer_id|customer_name|email               |status|updated_at         |source_system|last_merge_action|merge_processed_at        |
+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+
|C001       |Anan         |anan.new@example.com|active|2025-04-21 09:00:00|crm          |UPDATE           |2026-06-02 07:44:43.903493|
+-----------+-------------+--------------------+------+-------------------+-------------+-----------------+--------------------------+



### Exercise 3: Check target uniqueness after all merges

หลัง merge หลายรอบ ควรตรวจว่า target ยังมี 1 row ต่อ `customer_id`

Requirements:

- group by `customer_id`
- count records ต่อ key
- filter เฉพาะ key ที่มี `row_count > 1`

Expected result:

- ไม่ควรมี duplicate `customer_id` ใน target table

In [14]:
df_target_duplicate_keys = (
    spark.read.table(target_table_name)
    .groupBy("customer_id")
    .agg(F.count("*").alias("row_count"))
    .filter(F.col("row_count") > 1)
)

df_target_duplicate_keys.show(truncate=False)
print("Duplicate key count:", df_target_duplicate_keys.count())

StatementMeta(, 4c4b85e7-e7e4-4004-9cae-f99350f3d25f, 16, Finished, Available, Finished, False)

+-----------+---------+
|customer_id|row_count|
+-----------+---------+
+-----------+---------+

Duplicate key count: 0


## Common mistakes

1. **ใช้ `append` แทน MERGE กับข้อมูล snapshot/update**

   ถ้า source ส่ง record เดิมที่เปลี่ยนค่าเข้ามา การ append จะสร้าง duplicate business records แทนที่จะ update target row เดิม

2. **เลือก business key ผิด**

   ถ้า `ON` condition ไม่ถูกต้อง อาจ update ผิด record หรือ insert record ซ้ำ เช่นใช้ `email` เป็น key ทั้งที่ email เปลี่ยนได้

3. **ไม่ตรวจ duplicate source key ก่อน merge**

   Source batch ที่มี key ซ้ำควรถูก resolve ก่อน merge เช่น keep latest, reject หรือ apply business rule ที่ชัดเจน

4. **ยอมให้ stale update overwrite target**

   ถ้า source ส่งข้อมูลเก่ากว่า target เข้ามา และ merge ไม่มี condition เช่น `source.updated_at > target.updated_at` อาจทำให้ข้อมูลย้อนกลับเป็นค่าเก่า

5. **คิดว่า MERGE ทำให้ pipeline idempotent อัตโนมัติ**

   MERGE ช่วยเรื่อง update/insert แต่ idempotency ยังต้อง design เพิ่ม เช่น key uniqueness, source deduplication, update condition และ audit/check หลังรัน

6. **ไม่ตรวจผลหลัง merge**

   หลัง merge ควรตรวจ row count, updated records, inserted records และ duplicate key เสมอ โดยเฉพาะในช่วงพัฒนา pipeline


## Expected Output / Success Criteria

เมื่อจบ Day 15 ควรทำได้:

- อธิบายได้ว่า `MERGE` และ `UPSERT` ใช้แก้ปัญหาอะไรใน Lakehouse pipeline
- แยกได้ว่า record ไหนเป็น matched update และ record ไหนเป็น not matched insert
- สร้าง target Lakehouse table และ source update batch ด้วย PySpark ได้
- ตรวจ duplicate business key ใน source batch ก่อน merge ได้
- ใช้ `row_number()` เพื่อเลือก latest source record ต่อ key ได้
- ใช้ Spark SQL `MERGE INTO` เพื่อ update/insert target table ได้
- ใช้ conditional update เช่น `source.updated_at > target.updated_at` เพื่อกัน stale update ได้
- ตรวจ target row count และ business key uniqueness หลัง merge ได้
- อธิบายได้ว่า MERGE เป็นพื้นฐานของ incremental load, CDC-style update และ SCD pattern ในวันถัด ๆ ไป

## Final takeaway

MERGE / UPSERT คือ pattern ที่ทำให้ Lakehouse table รองรับข้อมูลที่เปลี่ยนแปลงได้ ไม่ใช่แค่รับข้อมูลใหม่แบบ append-only

> ก่อน merge ต้องรู้ business key, ตรวจ source batch รอบใหม่ให้เรียบร้อย และตรวจผลหลัง merge เสมอ

Mindset สำคัญของ Day 15 คือ:

- `append` เหมาะกับ event/history ที่เพิ่มอย่างเดียว
- `merge` เหมาะกับ latest state หรือ target ที่ต้อง update record เดิม
- source update batch ควร unique ต่อ business key ก่อน merge
- update condition ช่วยป้องกัน stale data overwrite
- rerun safety ไม่ได้เกิดเอง ต้อง design และตรวจด้วย code

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day15_customer_profile")